In [21]:
from entsoe import EntsoePandasClient
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv("/Users/raynapatel/Downloads/Zeus-Energy_Security_Index/datasets/entsoe/entsoe.env")
client = EntsoePandasClient(api_key="be9993ee-346c-45b9-bef1-201dd8f8bf3d")

start = pd.Timestamp("2021-01-01", tz="Europe/Berlin")
end   = pd.Timestamp("2024-12-31", tz="Europe/Berlin")

prices = client.query_day_ahead_prices("DE_LU", start=start, end=end)

os.makedirs("data/raw", exist_ok=True)
prices.to_csv("data/raw/day_ahead_prices_DE.csv")
print("Pulled successfully. Shape:", prices.shape)
print(prices.head(30))

Pulled successfully. Shape: (35037,)
2021-01-01 00:00:00+01:00    50.87
2021-01-01 01:00:00+01:00    48.19
2021-01-01 02:00:00+01:00    44.68
2021-01-01 03:00:00+01:00    42.92
2021-01-01 04:00:00+01:00    40.39
2021-01-01 05:00:00+01:00    40.20
2021-01-01 06:00:00+01:00    39.63
2021-01-01 07:00:00+01:00    40.09
2021-01-01 08:00:00+01:00    41.27
2021-01-01 09:00:00+01:00    44.88
2021-01-01 10:00:00+01:00    45.00
2021-01-01 11:00:00+01:00    47.20
2021-01-01 12:00:00+01:00    50.78
2021-01-01 13:00:00+01:00    45.49
2021-01-01 14:00:00+01:00    44.73
2021-01-01 15:00:00+01:00    46.59
2021-01-01 16:00:00+01:00    52.99
2021-01-01 17:00:00+01:00    60.26
2021-01-01 18:00:00+01:00    60.61
2021-01-01 19:00:00+01:00    60.36
2021-01-01 20:00:00+01:00    57.40
2021-01-01 21:00:00+01:00    53.86
2021-01-01 22:00:00+01:00    53.45
2021-01-01 23:00:00+01:00    49.72
2021-01-02 00:00:00+01:00    46.69
2021-01-02 01:00:00+01:00    42.43
2021-01-02 02:00:00+01:00    41.09
2021-01-02 03:00:0

In [22]:
import pandas as pd
import numpy as np
import os

# --- Load ---
df = pd.read_csv("data/raw/day_ahead_prices_DE.csv")
df.columns = ["timestamp", "price_eur_mwh"]

# --- Parse timestamps ---
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert("Europe/Berlin")
df = df.sort_values("timestamp").reset_index(drop=True)

# --- Check for missing values ---
print(f"Missing values: {df['price_eur_mwh'].isna().sum()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# --- Fill small gaps ---
df["price_eur_mwh"] = df["price_eur_mwh"].ffill(limit=3)

# --- Add time features ---
df["date"]       = df["timestamp"].dt.date
df["hour"]       = df["timestamp"].dt.hour
df["dayofweek"]  = df["timestamp"].dt.dayofweek
df["month"]      = df["timestamp"].dt.month
df["year"]       = df["timestamp"].dt.year
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

# --- Daily average ---
daily = df.groupby("date")["price_eur_mwh"].mean().reset_index()
daily.columns = ["date", "avg_price_eur_mwh"]
daily["date"] = pd.to_datetime(daily["date"])

# --- Save ---
os.makedirs("data/clean", exist_ok=True)
df.to_csv("data/clean/prices_hourly_DE.csv", index=False)
daily.to_csv("data/clean/prices_daily_DE.csv", index=False)

print("Saved. Shape:", daily.shape)
print(daily.head())

Missing values: 0
Date range: 2021-01-01 00:00:00+01:00 to 2024-12-30 23:00:00+01:00
Saved. Shape: (1460, 2)
        date  avg_price_eur_mwh
0 2021-01-01          48.398333
1 2021-01-02          50.562500
2 2021-01-03          38.622500
3 2021-01-04          48.017917
4 2021-01-05          55.337083
